In [12]:
from datasets import load_dataset
import numpy 
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    AutoModel,
    AutoTokenizer
)
import torch
from torch.utils.data import DataLoader


In [3]:
MODEL_NAME = "dumitrescustefan/bert-base-romanian-cased-v1"

In [ ]:
dataset = load_dataset("community-datasets/ronec")

label_list = dataset["train"].features["ner_tags"].feature.names



In [ ]:
id2label = {i: l for i, l in enumerate(label_list)}
num_labels = len(label_list)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [16]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation = True,
        max_length = 128,
        is_split_into_words = True
    )


    aligned_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index = i)

        prev_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None or word_idx == prev_word_idx:
                label_ids.append(-100)
            else:
                label_ids.append(word_labels[word_idx])
            prev_word_idx = word_idx

        aligned_labels.append(label_ids)
    tokenized["labels"] = aligned_labels
    return tokenized

In [17]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched = True,
    remove_columns = dataset["train"].column_names
)

collator = DataCollatorForTokenClassification(tokenizer = tokenizer)

# load train / valid / test ds using 
train_loader = DataLoader(tokenized_dataset["train"],
                        batch_size = 16,
                        shuffle = True,
                        collate_fn = collator)
val_loader = DataLoader(tokenized_dataset["validation"],
                        batch_size = 32,
                        shuffle = False,
                        collate_fn = collator)
test_loader = DataLoader(tokenized_dataset["test"],
                        batch_size = 32,
                        shuffle = False,
                        collate_fn = collator)

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1330 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [19]:
!pip install -q torchcrf

In [20]:
# custom model - robert + crf layer
from torchcrf import CRF

class RoBERT_CRF(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first = True)

    def _pack_valid_tokens(self, emissions, labels):
        """
        Extracts only real word-start tokens (labels != -100) per sentence and pads them,
        satisfying pytorch-crf's requirement that mask[:, 0] is always True and labels >= 0.
        """
        batch_size, _, num_tags = emissions.shape
        valid_emissions, valid_labels, lengths = [], [], []

        for b in range(batch_size):
            keep = labels[b] != -100
            valid_emissions.append(emissions[b][keep])
            valid_labels.append(labels[b][keep])
            lengths.append(keep.sum().item())

        max_len = max(max(lengths), 1)
        padded_emissions = torch.zeros(batch_size, max_len, num_tags, device=emissions.device)
        padded_labels = torch.zeros(batch_size, max_len, dtype=torch.long, device=emissions.device)
        mask = torch.zeros(batch_size, max_len, dtype=torch.bool, device=emissions.device)

        for b, length in enumerate(lengths):
            if length > 0:
                padded_emissions[b, :length] = valid_emissions[b]
                padded_labels[b, :length] = valid_labels[b]
                mask[b, :length] = True
            else:
                mask[b, 0] = True  # safety fallback for empty sequences

        return padded_emissions, padded_labels, mask

    def forward(self, input_ids, attention_mask, labels):
        seq_out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        emissions = self.classifier(self.dropout(seq_out))

        padded_emissions, padded_labels, mask = self._pack_valid_tokens(emissions, labels)
        # CRF returns log-likelihood -> negate for minimization loss
        loss = -self.crf(padded_emissions, padded_labels, mask=mask, reduction="mean")
        return loss

    @torch.no_grad()
    def decode(self, input_ids, attention_mask, labels):
        seq_out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        emissions = self.classifier(seq_out)
        padded_emissions, padded_labels, mask = self._pack_valid_tokens(emissions, labels)

        # Viterbi decoding returns a list of valid tag ID lists (one per sentence)
        best_paths = self.crf.decode(padded_emissions, mask=mask)
        gold_paths = [
            padded_labels[b][mask[b]].cpu().tolist()
            for b in range(len(best_paths))
        ]
        return best_paths, gold_paths


ModuleNotFoundError: No module named 'torchcrf'